In [1]:
import json
import pandas as pd
from pathlib import Path

# Path to the untouched ACN raw dataset
raw_path = Path("../Data/raw/acndata_sessions.json")

with open(raw_path, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

# Extract charging sessions and convert them to a DataFrame
df = pd.DataFrame(raw_data["_items"])

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (10609, 13)


,_id,clusterID,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs
0,5c3fda9df9af8b0d99c6fd60,0039,"Tue, 01 Jan 2019 03:45:49 GMT","Tue, 01 Jan 2019 17:57:23 GMT","Tue, 01 Jan 2019 10:40:45 GMT",43.114,2_39_138_29_2019-01-01 03:45:49.452098,0002,CA-304,2-39-138-29,America/Los_Angeles,000000489,"[{'WhPerMile': 475, 'kWhRequested': 99.75, 'mi..."
1,5c3fda9df9af8b0d99c6fd61,0039,"Tue, 01 Jan 2019 05:28:34 GMT","Tue, 01 Jan 2019 06:03:12 GMT","Tue, 01 Jan 2019 06:03:06 GMT",3.250,2_39_130_31_2019-01-01 05:28:33.527740,0002,CA-306,2-39-130-31,America/Los_Angeles,000001616,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile..."
2,5c412c1df9af8b12cb56c27c,0039,"Tue, 01 Jan 2019 17:41:45 GMT","Tue, 01 Jan 2019 18:39:21 GMT","Tue, 01 Jan 2019 18:40:21 GMT",0.900,2_39_138_566_2019-01-01 17:41:44.784919,0002,CA-512,2-39-138-566,America/Los_Angeles,NaN,None
3,5c412c1df9af8b12cb56c27d,0039,"Tue, 01 Jan 2019 18:09:17 GMT","Wed, 02 Jan 2019 02:39:32 GMT","Tue, 01 Jan 2019 20:16:10 GMT",12.534,2_39_79_379_2019-01-01 18:09:16.991864,0002,CA-327,2-39-79-379,America/Los_Angeles,000000558,"[{'WhPerMile': 222, 'kWhRequested': 17.76, 'mi..."
4,5c412c1df9af8b12cb56c27e,0039,"Tue, 01 Jan 2019 18:39:25 GMT","Tue, 01 Jan 2019 19:18:49 GMT","Tue, 01 Jan 2019 19:19:47 GMT",0.883,2_39_138_566_2019-01-01 18:39:24.566872,0002,CA-512,2-39-138-566,America/Los_Angeles,NaN,None


In [2]:
# Convert timestamp strings to datetime objects in UTC
time_columns = [
    "connectionTime",
    "disconnectTime",
    "doneChargingTime"
]

for column in time_columns:
    df[column] = pd.to_datetime(df[column], utc=True, errors="coerce")

# Create local California timestamps for behavioral/time-based analysis
df["connectionTime_local"] = df["connectionTime"].dt.tz_convert("America/Los_Angeles")
df["disconnectTime_local"] = df["disconnectTime"].dt.tz_convert("America/Los_Angeles")
df["doneChargingTime_local"] = df["doneChargingTime"].dt.tz_convert("America/Los_Angeles")

df[
    [
        "connectionTime",
        "connectionTime_local",
        "disconnectTime_local",
        "doneChargingTime_local"
    ]
].head()

,connectionTime,connectionTime_local,disconnectTime_local,doneChargingTime_local
0,2019-01-01 03:45:49+00:00,2018-12-31 19:45:49-08:00,2019-01-01 09:57:23-08:00,2019-01-01 02:40:45-08:00
1,2019-01-01 05:28:34+00:00,2018-12-31 21:28:34-08:00,2018-12-31 22:03:12-08:00,2018-12-31 22:03:06-08:00
2,2019-01-01 17:41:45+00:00,2019-01-01 09:41:45-08:00,2019-01-01 10:39:21-08:00,2019-01-01 10:40:21-08:00
3,2019-01-01 18:09:17+00:00,2019-01-01 10:09:17-08:00,2019-01-01 18:39:32-08:00,2019-01-01 12:16:10-08:00
4,2019-01-01 18:39:25+00:00,2019-01-01 10:39:25-08:00,2019-01-01 11:18:49-08:00,2019-01-01 11:19:47-08:00


In [3]:
# Calculate how long each EV remained connected
df["connected_hours"] = (
    df["disconnectTime"] - df["connectionTime"]
).dt.total_seconds() / 3600

# Calculate reported charging duration
df["charging_hours"] = (
    df["doneChargingTime"] - df["connectionTime"]
).dt.total_seconds() / 3600

# Negative charging duration is not physically meaningful
df.loc[df["charging_hours"] < 0, "charging_hours"] = pd.NA

print("Missing charging durations after cleaning:",
      df["charging_hours"].isna().sum())

df[["kWhDelivered", "connected_hours", "charging_hours"]].head()

Missing charging durations after cleaning: 473


,kWhDelivered,connected_hours,charging_hours
0,43.114,14.192778,6.915556
1,3.250,0.577222,0.575556
2,0.900,0.960000,0.976667
3,12.534,8.504167,2.114722
4,0.883,0.656667,0.672778


In [4]:
# Create time-based features from LOCAL Caltech time
df["date"] = df["connectionTime_local"].dt.date
df["hour"] = df["connectionTime_local"].dt.hour
df["day_of_week"] = df["connectionTime_local"].dt.dayofweek
df["day_name"] = df["connectionTime_local"].dt.day_name()
df["month"] = df["connectionTime_local"].dt.month

# Monday = 0 ... Sunday = 6
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df[
    [
        "connectionTime_local",
        "date",
        "hour",
        "day_of_week",
        "day_name",
        "month",
        "is_weekend"
    ]
].head()

,connectionTime_local,date,hour,day_of_week,day_name,month,is_weekend
0,2018-12-31 19:45:49-08:00,2018-12-31,19,0,Monday,12,0
1,2018-12-31 21:28:34-08:00,2018-12-31,21,0,Monday,12,0
2,2019-01-01 09:41:45-08:00,2019-01-01,9,1,Tuesday,1,0
3,2019-01-01 10:09:17-08:00,2019-01-01,10,1,Tuesday,1,0
4,2019-01-01 10:39:25-08:00,2019-01-01,10,1,Tuesday,1,0


In [5]:
# Keep only sessions whose connection started during 2019 in Caltech local time
df = df[
    (df["connectionTime_local"].dt.year == 2019)
].copy()

df.reset_index(drop=True, inplace=True)

print("Sessions after local-year filtering:", len(df))
print("First local connection:", df["connectionTime_local"].min())
print("Last local connection:", df["connectionTime_local"].max())

Sessions after local-year filtering: 10607
First local connection: 2019-01-01 09:41:45-08:00
Last local connection: 2019-12-30 15:29:51-08:00


In [6]:
processed_path = Path("../Data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

clean_file = processed_path / "acn_caltech_2019_clean.csv"

df.to_csv(clean_file, index=False)

print("Clean dataset saved to:", clean_file)
print("Final shape:", df.shape)

Clean dataset saved to: ..\Data\processed\acn_caltech_2019_clean.csv
Final shape: (10607, 24)
